In [ ]:
import random
import numpy as np
import tensorflow as tf

random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
import numpy as np
import pandas as pd
import random
import cv2
import os
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
from sklearn.metrics import accuracy_score

import tensorflow as tf
from tensorflow import expand_dims
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, Input, Lambda
from tensorflow.keras.layers import MaxPooling2D
from tensorflow.keras.layers import Activation
from tensorflow.keras.layers import Flatten
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import SGD
from tensorflow.keras import backend as K
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.applications.inception_v3 import preprocess_input
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.models import Model
from collections import defaultdict

In [ ]:
!pip install imutils
from imutils import paths

In [ ]:
# Dataset root (CONFIRMED from inspection)
DATASET_ROOT = "/kaggle/input/cifar10-pngs-in-folders/cifar10/cifar10"  #DATASET_ROOT = "path_to_cifar10_folder"

TRAIN_DIR = os.path.join(DATASET_ROOT, "train")
TEST_DIR  = os.path.join(DATASET_ROOT, "test")

# Get sorted class names
CLASSES = sorted(os.listdir(TRAIN_DIR))

print("Number of classes:", len(CLASSES))
print("Class names:", CLASSES)

# Pick one sample image for sanity check
sample_class = CLASSES[0]
sample_image_path = os.path.join(TRAIN_DIR, sample_class, os.listdir(os.path.join(TRAIN_DIR, sample_class))[0])

print("Sample image path:", sample_image_path)

# Load and preprocess one image (InceptionV3 expects 299x299 RGB)
img = load_img(sample_image_path, target_size=(299, 299))
img_array = img_to_array(img)
img_array = preprocess_input(img_array)

print("Image shape after preprocessing:", img_array.shape)


## Stage 1: Feature Extractor Backbone (InceptionV3)

In this step, we initialize a pretrained InceptionV3 model to act as a **feature extractor**.
The classification head is removed, and the backbone is kept frozen to focus on
few-shot learning dynamics rather than full supervised training.

The extracted embeddings will later be used to compute class prototypes
in the Prototypical Network framework.


In [ ]:
# =========================
# Stage 1: InceptionV3 Feature Extractor
# =========================

# Load pretrained InceptionV3 without the classification head
base_model = InceptionV3(
    weights="imagenet",
    include_top=False,
    input_shape=(299, 299, 3)
)

# Add a global average pooling layer to get fixed-size embeddings
feature_extractor = tf.keras.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D()
])

# Freeze all layers (important for few-shot setting)
for layer in feature_extractor.layers:
    layer.trainable = False

# Sanity check: pass one image through the feature extractor
sample_embedding = feature_extractor(
    tf.expand_dims(img_array, axis=0),
    training=False
)

print("Embedding shape:", sample_embedding.shape)


## Stage 1: Episodic Sampling for Few-Shot Learning

Few-shot learning is trained using an episodic paradigm.
Each episode simulates a small classification task by sampling:

- N classes (N-way)
- K samples per class for the support set (K-shot)
- Q samples per class for the query set

This setup allows the model to learn class-agnostic representations
that generalize to new tasks with limited labeled data.


In [ ]:
# =========================
# Stage 1: Episodic Sampler
# =========================

def sample_episode(
    train_dir,
    classes,
    n_way=5,
    k_shot=5,
    q_query=5
):
    """
    Samples a single N-way, K-shot episode.
    
    Returns:
        support_set: list of (image_path, class_index)
        query_set: list of (image_path, class_index)
    """
    selected_classes = random.sample(classes, n_way)
    
    support_set = []
    query_set = []
    
    for idx, cls in enumerate(selected_classes):
        cls_dir = os.path.join(train_dir, cls)
        images = os.listdir(cls_dir)
        
        selected_images = random.sample(images, k_shot + q_query)
        
        support_images = selected_images[:k_shot]
        query_images = selected_images[k_shot:]
        
        for img_name in support_images:
            support_set.append((os.path.join(cls_dir, img_name), idx))
        
        for img_name in query_images:
            query_set.append((os.path.join(cls_dir, img_name), idx))
    
    return support_set, query_set


In [ ]:
# Test episodic sampler
support_set, query_set = sample_episode(
    TRAIN_DIR,
    CLASSES,
    n_way=5,
    k_shot=5,
    q_query=5
)

print("Support set size:", len(support_set))
print("Query set size:", len(query_set))

print("First 3 support samples:", support_set[:3])
print("First 3 query samples:", query_set[:3])


## Stage 1: Prototypical Networks — Prototype Computation

In Prototypical Networks, each class is represented by a prototype,
computed as the mean embedding of its support examples.

Query samples are classified by measuring their distance to each class prototype
in the embedding space.


In [ ]:
# =========================
# Utility: Load image and extract embedding
# =========================

def get_embedding(image_path, model):
    img = load_img(image_path, target_size=(299, 299))
    img = img_to_array(img)
    img = preprocess_input(img)
    img = tf.expand_dims(img, axis=0)
    
    embedding = model(img, training=False)
    return tf.squeeze(embedding)


In [ ]:
# =========================
# Compute class prototypes
# =========================

def compute_prototypes(support_set, model):
    """
    support_set: list of (image_path, class_index)
    """
    embeddings_by_class = defaultdict(list)
    
    for image_path, class_idx in support_set:
        emb = get_embedding(image_path, model)
        embeddings_by_class[class_idx].append(emb)
    
    prototypes = {}
    for class_idx, embs in embeddings_by_class.items():
        prototypes[class_idx] = tf.reduce_mean(tf.stack(embs), axis=0)
    
    return prototypes


In [ ]:
# =========================
# Distance-based classification
# =========================

def classify_queries(query_set, prototypes, model):
    correct = 0
    total = len(query_set)
    
    for image_path, true_label in query_set:
        query_emb = get_embedding(image_path, model)
        
        distances = {
            class_idx: tf.norm(query_emb - proto)
            for class_idx, proto in prototypes.items()
        }
        
        predicted_label = min(distances, key=distances.get)
        
        if predicted_label == true_label:
            correct += 1
    
    return correct / total


In [ ]:
# Run a full few-shot episode
support_set, query_set = sample_episode(
    TRAIN_DIR,
    CLASSES,
    n_way=5,
    k_shot=5,
    q_query=5
)

prototypes = compute_prototypes(support_set, feature_extractor)
episode_accuracy = classify_queries(query_set, prototypes, feature_extractor)

print("Episode accuracy:", episode_accuracy)


## Stage 1: Few-Shot Evaluation Across Different Shot Settings

To evaluate the effect of data scarcity, we compare few-shot classification
performance under different shot settings:

- 1-shot
- 3-shot
- 5-shot

For each setting, multiple episodes are sampled and the average accuracy is reported.


In [ ]:
# =========================
# Evaluate few-shot performance across episodes
# =========================

def evaluate_few_shot(
    train_dir,
    classes,
    model,
    n_way=5,
    k_shot=5,
    q_query=5,
    num_episodes=20
):
    accuracies = []
    
    for episode in range(num_episodes):
        support_set, query_set = sample_episode(
            train_dir,
            classes,
            n_way=n_way,
            k_shot=k_shot,
            q_query=q_query
        )
        
        prototypes = compute_prototypes(support_set, model)
        acc = classify_queries(query_set, prototypes, model)
        accuracies.append(acc)
    
    return sum(accuracies) / len(accuracies)


In [ ]:
# Few-shot settings
SHOT_SETTINGS = [1, 3, 5]
RESULTS = {}

for k in SHOT_SETTINGS:
    avg_acc = evaluate_few_shot(
        TRAIN_DIR,
        CLASSES,
        feature_extractor,
        n_way=5,
        k_shot=k,
        q_query=5,
        num_episodes=20
    )
    
    RESULTS[k] = avg_acc
    print(f"{k}-shot average accuracy: {avg_acc:.4f}")


In [ ]:
import pandas as pd
import altair as alt

# Convert results to DataFrame
df = pd.DataFrame({
    "Shots (K)": list(RESULTS.keys()),
    "Average Accuracy": list(RESULTS.values())
})

chart = alt.Chart(df).mark_line(point=True).encode(
    x=alt.X("Shots (K):O", title="Number of Shots (K)"),
    y=alt.Y("Average Accuracy:Q", title="Average Accuracy"),
    tooltip=["Shots (K)", "Average Accuracy"]
).properties(
    title="Few-Shot Learning Performance (5-way CIFAR-10)",
    width=500,
    height=300
)

chart


In [ ]:
# Plot few-shot comparison
shots = list(RESULTS.keys())
accuracies = list(RESULTS.values())
import matplotlib.pyplot as plt
plt.figure()
plt.plot(shots, accuracies, marker='o')
plt.xlabel("Number of shots (K)")
plt.ylabel("Average Accuracy")
plt.title("Few-Shot Learning Performance (5-way CIFAR-10)")
plt.grid(True)
plt.show()


## Stage 2: Non-IID Client Data Partitioning

To simulate federated learning, the CIFAR-10 dataset is partitioned
into multiple non-overlapping client datasets.

Each client is assigned only a small subset of classes (2–3),
creating a strict non-IID data distribution across clients.
This setup reflects realistic federated scenarios where data
is heterogeneous and client-specific.


In [ ]:
# =========================
# Stage 2: Client definition and class allocation
# =========================

NUM_CLIENTS = 5

# Manually define non-IID class splits (2 classes per client)
CLIENT_CLASS_MAP = {
    0: ['airplane', 'automobile'],
    1: ['bird', 'cat'],
    2: ['deer', 'dog'],
    3: ['frog', 'horse'],
    4: ['ship', 'truck']
}

# Sanity check
for client_id, cls_list in CLIENT_CLASS_MAP.items():
    print(f"Client {client_id} classes: {cls_list}")


In [ ]:
# =========================
# Build client-local datasets
# =========================

def build_client_dataset(train_dir, class_list):
    image_paths = []
    labels = []
    
    for label_idx, cls in enumerate(class_list):
        cls_dir = os.path.join(train_dir, cls)
        for img_name in os.listdir(cls_dir):
            image_paths.append(os.path.join(cls_dir, img_name))
            labels.append(label_idx)
    
    return image_paths, labels


CLIENT_DATASETS = {}

for client_id, class_list in CLIENT_CLASS_MAP.items():
    paths, labels = build_client_dataset(TRAIN_DIR, class_list)
    CLIENT_DATASETS[client_id] = (paths, labels)
    print(f"Client {client_id}: {len(paths)} samples")


## Stage 2: Local Client Training

In this step, each federated client performs local training on its own
private dataset. The training process is fully independent across clients,
and only model updates (not data) will later be shared with the server.

This setup simulates the core principle of federated learning:
data locality with decentralized training.


In [ ]:
# =========================
# Stage 2: Client dataset loader
# =========================

def create_tf_dataset(image_paths, labels, batch_size=32, shuffle=True):
    def load_and_preprocess(path, label):
        img = load_img(path.numpy().decode(), target_size=(299, 299))
        img = img_to_array(img)
        img = preprocess_input(img)
        return img, label

    def tf_wrapper(path, label):
        img, lbl = tf.py_function(
            load_and_preprocess,
            inp=[path, label],
            Tout=[tf.float32, tf.int32]
        )
        img.set_shape((299, 299, 3))
        lbl.set_shape(())
        return img, lbl

    dataset = tf.data.Dataset.from_tensor_slices((image_paths, labels))
    if shuffle:
        dataset = dataset.shuffle(buffer_size=len(image_paths))
    dataset = dataset.map(tf_wrapper, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return dataset


In [ ]:
# =========================
# Stage 2: Client model definition
# =========================

def create_client_model(num_classes):
    base = InceptionV3(
        weights="imagenet",
        include_top=False,
        input_shape=(299, 299, 3)
    )
    base.trainable = False  # freeze backbone

    x = tf.keras.layers.GlobalAveragePooling2D()(base.output)
    output = tf.keras.layers.Dense(num_classes, activation="softmax")(x)

    model = tf.keras.Model(inputs=base.input, outputs=output)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


In [ ]:
# =========================
# Stage 2: Local client training
# =========================

def train_client(client_id, client_data, epochs=1, batch_size=32):
    image_paths, labels = client_data
    dataset = create_tf_dataset(image_paths, labels, batch_size=batch_size)

    num_classes = len(set(labels))
    model = create_client_model(num_classes)

    history = model.fit(dataset, epochs=epochs, verbose=0)
    
    return model.get_weights(), history.history["accuracy"][-1]


In [ ]:
# Test local training on a single client
client_id = 0
client_data = CLIENT_DATASETS[client_id]

weights, acc = train_client(client_id, client_data, epochs=1)

print(f"Client {client_id} local training accuracy: {acc:.4f}")


## Stage 2: Federated Averaging (FedAvg)

In this step, we simulate federated learning using the FedAvg algorithm.
A global model is initialized and shared with all clients.

Each client:
- Receives the global model
- Trains locally on private data
- Sends updated weights back to the server

The server aggregates client updates by averaging the weights,
forming a new global model for the next communication round.


In [ ]:
# =========================
# Stage 2: FedAvg aggregation
# =========================

def fedavg_aggregate(client_weights):
    """
    client_weights: list of model weight lists
    """
    avg_weights = []
    
    for weights in zip(*client_weights):
        avg_weights.append(tf.reduce_mean(tf.stack(weights), axis=0))
    
    return avg_weights


In [ ]:
# =========================
# Initialize global model
# =========================

GLOBAL_NUM_CLASSES = 2  # each client has 2 classes

global_model = create_client_model(GLOBAL_NUM_CLASSES)


In [ ]:
# =========================
# Stage 2: Federated training loop (with logging)
# =========================

import numpy as np
import pandas as pd

NUM_ROUNDS = 5
LOCAL_EPOCHS = 1

# ---- storage ----
global_accuracies = []          # avg accuracy per round
round_client_accuracies = []    # per-client accuracy per round
round_global_weights = []       # optional: global weights per round
training_log = []               # structured log (report-friendly)

for round_idx in range(NUM_ROUNDS):
    print(f"\n--- Federated Round {round_idx + 1} ---")
    
    client_weights = []
    client_accuracies = {}
    
    # Send global model to clients
    global_weights = global_model.get_weights()
    
    for client_id, client_data in CLIENT_DATASETS.items():
        # Initialize client model
        client_model = create_client_model(GLOBAL_NUM_CLASSES)
        client_model.set_weights(global_weights)
        
        # Local dataset
        dataset = create_tf_dataset(
            client_data[0],
            client_data[1],
            batch_size=32
        )
        
        # Local training
        history = client_model.fit(
            dataset,
            epochs=LOCAL_EPOCHS,
            verbose=0
        )
        
        acc = history.history["accuracy"][-1]
        
        client_weights.append(client_model.get_weights())
        client_accuracies[client_id] = acc
        
        print(f"Client {client_id} accuracy: {acc:.4f}")
    
    # ---- FedAvg aggregation ----
    new_global_weights = fedavg_aggregate(client_weights)
    global_model.set_weights(new_global_weights)
    
    avg_client_acc = sum(client_accuracies.values()) / len(client_accuracies)
    
    # ---- store results ----
    global_accuracies.append(avg_client_acc)
    round_client_accuracies.append(client_accuracies)
    round_global_weights.append(new_global_weights)
    
    training_log.append({
        "round": round_idx + 1,
        "client_accuracies": client_accuracies,
        "avg_client_accuracy": avg_client_acc
    })
    
    print(f"Average client accuracy (round {round_idx + 1}): {avg_client_acc:.4f}")

# =========================
# Save results to disk
# =========================

# Per-client accuracy CSV
rows = []
for r, client_accs in enumerate(round_client_accuracies):
    for client_id, acc in client_accs.items():
        rows.append({
            "round": r + 1,
            "client": client_id,
            "accuracy": acc
        })

df = pd.DataFrame(rows)
df.to_csv("fedavg_client_accuracies.csv", index=False)

# Global accuracy per round
np.save("fedavg_global_accuracies.npy", np.array(global_accuracies))

print("\nTraining complete. Results saved.")

In [ ]:
import pandas as pd
import altair as alt

# -------------------------
# Load saved federated results
# -------------------------
df = pd.read_csv("fedavg_client_accuracies.csv")

# Optional: inspect the data
df
# -------------------------
# Heatmap visualization
# -------------------------
heatmap = alt.Chart(df).mark_rect().encode(
    x=alt.X("round:O", title="Federated Round"),
    y=alt.Y("client:O", title="Client"),
    color=alt.Color(
        "accuracy:Q",
        scale=alt.Scale(scheme="viridis"),
        title="Accuracy"
    ),
    tooltip=["round", "client", "accuracy"]
).properties(
    title="Federated Learning: Client Accuracy Heatmap",
    width=400,
    height=250
)

heatmap

In [ ]:
import pandas as pd
import altair as alt

# -------------------------
# Load saved federated results
# -------------------------
df = pd.read_csv("fedavg_client_accuracies.csv")

# -------------------------
# Line chart visualization
# -------------------------
line_chart = alt.Chart(df).mark_line(point=True).encode(
    x=alt.X("round:O", title="Federated Round"),
    y=alt.Y(
        "accuracy:Q",
        scale=alt.Scale(domain=[0.90, 1.0]),
        title="Accuracy"
    ),
    color=alt.Color("client:N", title="Client"),
    tooltip=["client", "round", "accuracy"]
).properties(
    title="Client-wise Accuracy Progression Across Federated Rounds",
    width=450,
    height=280
)

line_chart

In [ ]:
plt.figure()
plt.plot(range(1, NUM_ROUNDS + 1), global_accuracies, marker='o')
plt.xlabel("Federated Rounds")
plt.ylabel("Average Client Accuracy")
plt.title("Federated Learning Convergence (FedAvg)")
plt.grid(True)
plt.show()


## Stage 3: Few-Shot Federated Clients

In this stage, each federated client operates under a few-shot constraint.
Rather than training a full classifier, each client computes class prototypes
from a limited number of local samples.

Only these prototypes are shared with the server, significantly reducing
communication cost and enabling few-shot federated learning.


In [ ]:
# =========================
# Stage 3: Few-shot client data creation
# =========================

def create_few_shot_client_data(client_dataset, k_shot=5):
    image_paths, labels = client_dataset
    
    data_by_class = defaultdict(list)
    for path, label in zip(image_paths, labels):
        data_by_class[label].append(path)
    
    few_shot_data = []
    for label, paths in data_by_class.items():
        selected = random.sample(paths, k_shot)
        for p in selected:
            few_shot_data.append((p, label))
    
    return few_shot_data


In [ ]:
# Test few-shot client data
client_id = 0
few_shot_data = create_few_shot_client_data(CLIENT_DATASETS[client_id], k_shot=5)

print("Few-shot samples:", len(few_shot_data))
print("Samples per class:", len(few_shot_data) // 2)


## Stage 3: Local Prototype Computation at Clients

In few-shot federated learning, each client summarizes its local data
by computing class prototypes in the embedding space.

A prototype is defined as the mean embedding of all samples belonging
to a class. Only these prototypes are shared with the server, enabling
communication-efficient federated learning under data scarcity.


In [ ]:
# =========================
# Stage 3: Client-side prototype computation
# =========================

def compute_client_prototypes(few_shot_data, model):
    """
    few_shot_data: list of (image_path, class_label)
    Returns:
        dict: class_label -> prototype tensor
    """
    embeddings_by_class = defaultdict(list)
    
    for image_path, label in few_shot_data:
        emb = get_embedding(image_path, model)
        embeddings_by_class[label].append(emb)
    
    client_prototypes = {}
    for label, embs in embeddings_by_class.items():
        client_prototypes[label] = tf.reduce_mean(tf.stack(embs), axis=0)
    
    return client_prototypes


In [ ]:
# Test prototype computation for one client
client_id = 0
few_shot_data = create_few_shot_client_data(
    CLIENT_DATASETS[client_id],
    k_shot=5
)

client_prototypes = compute_client_prototypes(
    few_shot_data,
    feature_extractor
)

for label, proto in client_prototypes.items():
    print(f"Client {client_id}, class {label}, prototype shape: {proto.shape}")


## Stage 3: Server-Side Prototype Aggregation

In prototype-based federated learning, the server aggregates class prototypes
received from multiple clients instead of model weights.

For each class, the global prototype is computed as the mean of all
client-provided prototypes corresponding to that class.


In [ ]:
# =========================
# Stage 3: Server-side prototype aggregation
# =========================

def aggregate_prototypes(client_prototype_list):
    """
    client_prototype_list: list of dicts (class_label -> prototype)
    Returns:
        dict: class_label -> global prototype
    """
    proto_by_class = defaultdict(list)
    
    for client_proto in client_prototype_list:
        for label, proto in client_proto.items():
            proto_by_class[label].append(proto)
    
    global_prototypes = {}
    for label, protos in proto_by_class.items():
        global_prototypes[label] = tf.reduce_mean(
            tf.stack(protos), axis=0
        )
    
    return global_prototypes


In [ ]:
# Collect prototypes from all clients
all_client_prototypes = []

for client_id in CLIENT_DATASETS.keys():
    few_shot_data = create_few_shot_client_data(
        CLIENT_DATASETS[client_id],
        k_shot=5
    )
    
    client_proto = compute_client_prototypes(
        few_shot_data,
        feature_extractor
    )
    
    all_client_prototypes.append(client_proto)

# Aggregate at server
global_prototypes = aggregate_prototypes(all_client_prototypes)

for label, proto in global_prototypes.items():
    print(f"Global prototype for class {label}: shape {proto.shape}")


## Stage 3: Global Evaluation and Comparison

In this final step, we evaluate the prototype-based federated model
on unseen query samples and compare its performance against the
standard FedAvg baseline.

This comparison highlights the benefits of prototype-based communication
in few-shot federated learning scenarios.


In [ ]:
# =========================
# Stage 3: Prototype-based global evaluation
# =========================

def evaluate_with_global_prototypes(
    test_dir,
    global_prototypes,
    model,
    num_samples_per_class=20
):
    correct = 0
    total = 0
    
    for cls_name, cls_label in zip(CLASSES, range(len(CLASSES))):
        cls_dir = os.path.join(test_dir, cls_name)
        images = random.sample(os.listdir(cls_dir), num_samples_per_class)
        
        for img_name in images:
            img_path = os.path.join(cls_dir, img_name)
            emb = get_embedding(img_path, model)
            
            # Compute distance to global prototypes
            distances = {
                label: tf.norm(emb - proto)
                for label, proto in global_prototypes.items()
            }
            
            predicted_label = min(distances, key=distances.get)
            
            if predicted_label == cls_label:
                correct += 1
            total += 1
    
    return correct / total


In [ ]:
# Run evaluation
few_shot_fl_accuracy = evaluate_with_global_prototypes(
    TEST_DIR,
    global_prototypes,
    feature_extractor,
    num_samples_per_class=20
)

print("Few-Shot Federated Learning Accuracy:", few_shot_fl_accuracy)


# Stage 3: Multi-Round Prototype Federated Learning

While the previous experiment evaluates few-shot federated learning using a single round of prototype aggregation, real federated systems typically involve multiple communication rounds between clients and the server. To better reflect this setting, we extend the prototype-based federated learning framework to a multi-round scenario.

In this experiment, federated learning proceeds over multiple communication rounds. During each round, every client independently samples a few-shot subset of its local data and computes class prototypes using a frozen feature extractor. These client-side prototypes are then transmitted to the server, where they are aggregated via simple averaging to form global class prototypes. The updated global prototypes are broadcast back to clients in the next round.

Importantly, no model parameters are trained or updated during this process. Communication is restricted exclusively to prototype representations, preserving the communication-efficient and privacy-aware nature of the proposed approach.

The global model is evaluated after each communication round using a held-out test set. This allows us to analyze convergence behavior and assess whether repeated prototype aggregation improves global performance under extreme few-shot and non-IID constraints.

Experimental results show that global accuracy stabilizes quickly across rounds, fluctuating within a narrow range around 18–19%. This behavior indicates that, in the absence of shared representation learning, additional communication rounds alone are insufficient to significantly improve performance. These findings highlight a fundamental limitation of prototype-only federated learning under severe data scarcity and motivate the need for improved aggregation strategies or federated representation learning in practical deployments

In [ ]:
# ============================================================
# Stage 3: Multi-Round Prototype Federated Learning
# ============================================================

NUM_PROTO_ROUNDS = 10
K_SHOT = 5
SAMPLES_PER_CLASS_TEST = 20

# ---------- INITIAL PROTOTYPE AGGREGATION (ROUND 0) ----------
all_client_prototypes = []

for client_id in CLIENT_DATASETS:
    few_shot_data = create_few_shot_client_data(
        CLIENT_DATASETS[client_id],
        k_shot=K_SHOT
    )
    
    client_proto = compute_client_prototypes(
        few_shot_data,
        feature_extractor
    )
    
    all_client_prototypes.append(client_proto)

global_prototypes = aggregate_prototypes(all_client_prototypes)

# ---------- MULTI-ROUND FEDERATED PROTOTYPE LEARNING ----------
global_accuracy_per_round = []

for round_idx in range(NUM_PROTO_ROUNDS):
    print(f"\n--- Prototype FL Round {round_idx + 1} ---")
    
    all_client_prototypes = []
    
    # ----- CLIENT SIDE -----
    for client_id in CLIENT_DATASETS:
        few_shot_data = create_few_shot_client_data(
            CLIENT_DATASETS[client_id],
            k_shot=K_SHOT
        )
        
        # Prototype computation using frozen feature extractor
        client_proto = compute_client_prototypes(
            few_shot_data,
            feature_extractor
        )
        
        all_client_prototypes.append(client_proto)
    
    # ----- SERVER SIDE -----
    global_prototypes = aggregate_prototypes(all_client_prototypes)
    
    # ----- GLOBAL EVALUATION -----
    acc = evaluate_with_global_prototypes(
        TEST_DIR,
        global_prototypes,
        feature_extractor,
        num_samples_per_class=SAMPLES_PER_CLASS_TEST
    )
    
    global_accuracy_per_round.append(acc)
    print(f"Global accuracy (round {round_idx + 1}): {acc:.4f}")

# ---------- FINAL RESULT ----------
final_accuracy = global_accuracy_per_round[-1]
print(
    f"\nFinal Few-Shot Federated Learning Accuracy "
    f"after {NUM_PROTO_ROUNDS} rounds: {final_accuracy:.4f}"
)


In [ ]:


rounds = list(range(1, len(global_accuracy_per_round) + 1))

plt.figure()
plt.plot(rounds, global_accuracy_per_round, marker='o')
plt.xlabel("Communication Round")
plt.ylabel("Global Accuracy")
plt.title("Multi-Round Few-Shot Federated Learning (InceptionV3)")
plt.grid(True)
plt.savefig("fig_global_accuracy_inception.png", dpi=300, bbox_inches="tight")

plt.show()


# Client-Wise Accuracy (Multi-Round FL)
Client-Wise Performance Analysis

In federated learning, global accuracy alone may obscure heterogeneous client behavior, particularly under non-IID data distributions. To better understand the impact of prototype-based aggregation on individual participants, we additionally report client-wise accuracy at each communication round.

After each round of prototype aggregation, the global prototypes are evaluated separately on each client’s local data. This analysis reveals notable variability in performance across clients, reflecting differences in class separability and data characteristics. While some clients consistently benefit from global prototype aggregation, others experience limited or fluctuating gains.

These observations highlight an important limitation of prototype-only federated learning: although communication-efficient, it does not guarantee uniform performance improvements across clients. This reinforces the need for client-aware aggregation strategies or personalized adaptation mechanisms in practical few-shot federated systems.

In [ ]:
def evaluate_client_with_global_prototypes(
    client_dataset,
    global_prototypes,
    model,
    num_samples_per_class=20
):
    image_paths, labels = client_dataset
    
    data_by_class = defaultdict(list)
    for path, label in zip(image_paths, labels):
        data_by_class[label].append(path)
    
    correct = 0
    total = 0
    
    for label, paths in data_by_class.items():
        sampled = random.sample(
            paths,
            min(num_samples_per_class, len(paths))
        )
        
        for img_path in sampled:
            emb = get_embedding(img_path, model)
            
            distances = {
                proto_label: tf.norm(emb - proto)
                for proto_label, proto in global_prototypes.items()
            }
            
            predicted = min(distances, key=distances.get)
            if predicted == label:
                correct += 1
            total += 1
    
    return correct / total if total > 0 else 0.0


In [ ]:
NUM_PROTO_ROUNDS = 10
K_SHOT = 5
SAMPLES_PER_CLASS_TEST = 20

# ---------- INITIAL PROTOTYPE AGGREGATION (ROUND 0) ----------
all_client_prototypes = []

for client_id in CLIENT_DATASETS:
    few_shot_data = create_few_shot_client_data(
        CLIENT_DATASETS[client_id],
        k_shot=K_SHOT
    )
    
    client_proto = compute_client_prototypes(
        few_shot_data,
        feature_extractor
    )
    
    all_client_prototypes.append(client_proto)

global_prototypes = aggregate_prototypes(all_client_prototypes)


client_accuracy_per_round = []

for round_idx in range(NUM_PROTO_ROUNDS):
    print(f"\n--- Prototype FL Round {round_idx + 1} ---")
    
    all_client_prototypes = []
    
    for client_id in CLIENT_DATASETS:
        few_shot_data = create_few_shot_client_data(
            CLIENT_DATASETS[client_id],
            k_shot=K_SHOT
        )
        
        client_proto = compute_client_prototypes(
            few_shot_data,
            feature_extractor
        )
        all_client_prototypes.append(client_proto)
    
    global_prototypes = aggregate_prototypes(all_client_prototypes)
    
    # ----- GLOBAL ACCURACY -----
    global_acc = evaluate_with_global_prototypes(
        TEST_DIR,
        global_prototypes,
        feature_extractor,
        num_samples_per_class=SAMPLES_PER_CLASS_TEST
    )
    
    # ----- CLIENT-WISE ACCURACY (NEW) -----
    round_client_accs = []
    
    for client_id, client_data in CLIENT_DATASETS.items():
        acc = evaluate_client_with_global_prototypes(
            client_data,
            global_prototypes,
            feature_extractor,
            num_samples_per_class=SAMPLES_PER_CLASS_TEST
        )
        round_client_accs.append(acc)
        print(f"Client {client_id} accuracy: {acc:.4f}")
    
    client_accuracy_per_round.append(round_client_accs)
    
    print(
        f"Global accuracy: {global_acc:.4f} | "
        f"Client mean ± std: "
        f"{np.mean(round_client_accs):.4f} ± {np.std(round_client_accs):.4f}"
    )


In [ ]:
num_clients = len(client_accuracy_per_round[0])
num_rounds = len(client_accuracy_per_round)

rounds = list(range(1, num_rounds + 1))

plt.figure(figsize=(8, 5))

for client_id in range(num_clients):
    client_curve = []
    for r in range(num_rounds):
        client_curve.append(client_accuracy_per_round[r][client_id])
    
    plt.plot(
        rounds,
        client_curve,
        marker='o',
        linewidth=2,
        label=f"Client {client_id}"
    )

plt.xlabel("Communication Round")
plt.ylabel("Client Accuracy")
plt.title("Client-Wise Accuracy Across Federated Rounds")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig("fig_client_wise_accuracy.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
import numpy as np

client_means = [
    np.mean(round_accs)
    for round_accs in client_accuracy_per_round
]

client_stds = [
    np.std(round_accs)
    for round_accs in client_accuracy_per_round
]

plt.figure()
plt.plot(rounds, client_means, marker='o')
plt.fill_between(
    rounds,
    np.array(client_means) - np.array(client_stds),
    np.array(client_means) + np.array(client_stds),
    alpha=0.2
)
plt.xlabel("Communication Round")
plt.ylabel("Client Accuracy")
plt.title("Mean ± Std Client Accuracy Across Rounds")
plt.grid(True)
plt.savefig("fig_client_mean_std.png", dpi=300, bbox_inches="tight")
plt.show()


# Backbone Comparison (InceptionV3 vs ResNet50)
Effect of Backbone Architecture

To evaluate the impact of representation quality on few-shot federated learning, we repeat the multi-round prototype aggregation experiments using two different pretrained feature extractors: InceptionV3 and ResNet50. In both cases, the backbone networks are kept frozen and only class prototypes are communicated between clients and the server.

Results indicate that ResNet50 provides a modest improvement in global accuracy compared to InceptionV3, suggesting that stronger representations can partially mitigate the challenges of few-shot federated learning. However, performance saturation persists across communication rounds, confirming that representation quality alone does not fully address the limitations of prototype-only aggregation under extreme data scarcity.

In [ ]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess

def create_resnet_feature_extractor():
    base = ResNet50(
        weights="imagenet",
        include_top=False,
        input_shape=(224, 224, 3)
    )
    base.trainable = False
    
    x = tf.keras.layers.GlobalAveragePooling2D()(base.output)
    model = tf.keras.Model(inputs=base.input, outputs=x)
    return model

resnet_feature_extractor = create_resnet_feature_extractor()


In [ ]:
def get_resnet_embedding(image_path, model):
    img = load_img(image_path, target_size=(224, 224))
    img = img_to_array(img)
    img = resnet_preprocess(img)
    img = tf.expand_dims(img, axis=0)
    emb = model(img, training=False)
    return tf.squeeze(emb)


In [ ]:
def compute_client_prototypes_resnet(few_shot_data, model):
    proto_by_class = defaultdict(list)
    
    for img_path, label in few_shot_data:
        emb = get_resnet_embedding(img_path, model)
        proto_by_class[label].append(emb)
    
    return {
        label: tf.reduce_mean(tf.stack(embs), axis=0)
        for label, embs in proto_by_class.items()
    }


In [ ]:
# ============================================================
# Stage 3: Multi-Round Prototype FL with ResNet50
# ============================================================

NUM_PROTO_ROUNDS = 10
K_SHOT = 5
SAMPLES_PER_CLASS_TEST = 20

resnet_accuracy_per_round = []

for round_idx in range(NUM_PROTO_ROUNDS):
    print(f"\n--- ResNet50 Prototype FL Round {round_idx + 1} ---")
    
    all_client_prototypes = []
    
    # ---------- CLIENT SIDE ----------
    for client_id in CLIENT_DATASETS:
        few_shot_data = create_few_shot_client_data(
            CLIENT_DATASETS[client_id],
            k_shot=K_SHOT
        )
        
        client_proto = compute_client_prototypes_resnet(
            few_shot_data,
            resnet_feature_extractor
        )
        all_client_prototypes.append(client_proto)
    
    # ---------- SERVER SIDE ----------
    global_prototypes_resnet = aggregate_prototypes(
        all_client_prototypes
    )
    
    # ---------- GLOBAL EVALUATION ----------
    correct, total = 0, 0
    
    for cls_name, cls_label in zip(CLASSES, range(len(CLASSES))):
        cls_dir = os.path.join(TEST_DIR, cls_name)
        images = random.sample(
            os.listdir(cls_dir),
            SAMPLES_PER_CLASS_TEST
        )
        
        for img_name in images:
            img_path = os.path.join(cls_dir, img_name)
            emb = get_resnet_embedding(
                img_path,
                resnet_feature_extractor
            )
            
            distances = {
                label: tf.norm(emb - proto)
                for label, proto in global_prototypes_resnet.items()
            }
            
            predicted = min(distances, key=distances.get)
            if predicted == cls_label:
                correct += 1
            total += 1
    
    acc = correct / total
    resnet_accuracy_per_round.append(acc)
    
    print(f"Global accuracy (round {round_idx + 1}): {acc:.4f}")

print(
    f"\nFinal ResNet50 Few-Shot FL Accuracy: "
    f"{resnet_accuracy_per_round[-1]:.4f}"
)


In [ ]:
rounds = list(range(1, len(global_accuracy_per_round) + 1))

plt.figure()
plt.plot(rounds, global_accuracy_per_round, marker='o', label="InceptionV3")
plt.plot(rounds, resnet_accuracy_per_round, marker='o', label="ResNet50")
plt.xlabel("Communication Round")
plt.ylabel("Global Accuracy")
plt.title("Backbone Comparison in Multi-Round Few-Shot FL")
plt.legend()
plt.grid(True)
plt.savefig("fig_backbone_comparison.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
final_inception_acc = global_accuracy_per_round[-1]
final_resnet_acc = resnet_accuracy_per_round[-1]

plt.figure()
plt.bar(
    ["InceptionV3", "ResNet50"],
    [final_inception_acc, final_resnet_acc]
)
plt.ylabel("Final Global Accuracy")
plt.title("Final Accuracy Comparison (Few-Shot FL)")
plt.grid(axis='y')
plt.savefig("fig_final_accuracy_bar.png", dpi=300, bbox_inches="tight")

plt.show()



### Stage 3 — Experimental Variants (Ablation & Improvement Study)



In [ ]:
# ============================================================
# Stage 3 - Option B: Improved Few-Shot Federated Learning
# (Higher k-shot + Weighted Prototype Aggregation)
# ============================================================

K_SHOT_IMPROVED = 10   # increased from 5
SAMPLES_PER_CLASS_TEST = 20

def compute_client_prototypes_with_counts(few_shot_data, model):
    proto_sum = {}
    proto_count = {}
    
    for image_path, label in few_shot_data:
        emb = get_embedding(image_path, model)
        if label not in proto_sum:
            proto_sum[label] = emb
            proto_count[label] = 1
        else:
            proto_sum[label] += emb
            proto_count[label] += 1
    
    return proto_sum, proto_count


def weighted_aggregate_prototypes(client_proto_info):
    global_sum = defaultdict(lambda: 0)
    global_count = defaultdict(int)
    
    for proto_sum, proto_count in client_proto_info:
        for label in proto_sum:
            global_sum[label] += proto_sum[label]
            global_count[label] += proto_count[label]
    
    global_prototypes = {}
    for label in global_sum:
        global_prototypes[label] = global_sum[label] / global_count[label]
    
    return global_prototypes


# ----- CLIENT SIDE -----
client_proto_info = []

for client_id in CLIENT_DATASETS:
    few_shot_data = create_few_shot_client_data(
        CLIENT_DATASETS[client_id],
        k_shot=K_SHOT_IMPROVED
    )
    
    proto_sum, proto_count = compute_client_prototypes_with_counts(
        few_shot_data,
        feature_extractor
    )
    
    client_proto_info.append((proto_sum, proto_count))


# ----- SERVER SIDE -----
global_prototypes_option_b = weighted_aggregate_prototypes(client_proto_info)


# ----- EVALUATION -----
correct, total = 0, 0

for cls_name, cls_label in zip(CLASSES, range(len(CLASSES))):
    cls_dir = os.path.join(TEST_DIR, cls_name)
    images = random.sample(os.listdir(cls_dir), SAMPLES_PER_CLASS_TEST)
    
    for img_name in images:
        img_path = os.path.join(cls_dir, img_name)
        emb = get_embedding(img_path, feature_extractor)
        
        distances = {
            label: tf.norm(emb - proto)
            for label, proto in global_prototypes_option_b.items()
        }
        
        predicted = min(distances, key=distances.get)
        if predicted == cls_label:
            correct += 1
        total += 1

option_b_accuracy = correct / total
print(f"Option B (Improved Few-Shot FL) Accuracy: {option_b_accuracy:.4f}")


In [ ]:
# ============================================================
# Stage 3 - Option C: Few-Shot FL with Local Fine-Tuning
# (Partial backbone adaptation at clients)
# ============================================================

K_SHOT = 5
LOCAL_EPOCHS = 3
FINE_TUNE_LAYERS = 20   # unfreeze last N layers
SAMPLES_PER_CLASS_TEST = 20


def create_finetunable_feature_extractor():
    base = InceptionV3(
        weights="imagenet",
        include_top=False,
        input_shape=(299, 299, 3)
    )
    
    # Freeze all layers first
    for layer in base.layers:
        layer.trainable = False
    
    # Unfreeze last few layers
    for layer in base.layers[-FINE_TUNE_LAYERS:]:
        layer.trainable = True
    
    x = tf.keras.layers.GlobalAveragePooling2D()(base.output)
    model = tf.keras.Model(inputs=base.input, outputs=x)
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-4),
        loss=None  # no supervised loss, embedding learning only
    )
    
    return model


def fine_tune_on_few_shot(model, few_shot_data, epochs=3):
    images = []
    labels = []
    
    for img_path, label in few_shot_data:
        img = load_img(img_path, target_size=(299, 299))
        img = img_to_array(img)
        img = preprocess_input(img)
        images.append(img)
        labels.append(label)
    
    images = tf.convert_to_tensor(images)
    labels = tf.convert_to_tensor(labels)
    
    # Dummy self-supervised objective: encourage compact embeddings
    with tf.GradientTape() as tape:
        embeddings = model(images, training=True)
        loss = tf.reduce_mean(tf.math.reduce_variance(embeddings, axis=0))
    
    grads = tape.gradient(loss, model.trainable_variables)
    model.optimizer.apply_gradients(zip(grads, model.trainable_variables))


# ---------- CLIENT SIDE ----------
client_prototypes = []

for client_id in CLIENT_DATASETS:
    few_shot_data = create_few_shot_client_data(
        CLIENT_DATASETS[client_id],
        k_shot=K_SHOT
    )
    
    local_model = create_finetunable_feature_extractor()
    
    # Local adaptation
    for _ in range(LOCAL_EPOCHS):
        fine_tune_on_few_shot(local_model, few_shot_data)
    
    # Compute prototypes after adaptation
    proto_by_class = defaultdict(list)
    for img_path, label in few_shot_data:
        emb = get_embedding(img_path, local_model)
        proto_by_class[label].append(emb)
    
    client_proto = {
        label: tf.reduce_mean(tf.stack(embs), axis=0)
        for label, embs in proto_by_class.items()
    }
    
    client_prototypes.append(client_proto)


# ---------- SERVER SIDE ----------
global_prototypes_option_c = aggregate_prototypes(client_prototypes)


# ---------- EVALUATION ----------
correct, total = 0, 0

for cls_name, cls_label in zip(CLASSES, range(len(CLASSES))):
    cls_dir = os.path.join(TEST_DIR, cls_name)
    images = random.sample(os.listdir(cls_dir), SAMPLES_PER_CLASS_TEST)
    
    for img_name in images:
        img_path = os.path.join(cls_dir, img_name)
        emb = get_embedding(img_path, local_model)
        
        distances = {
            label: tf.norm(emb - proto)
            for label, proto in global_prototypes_option_c.items()
        }
        
        predicted = min(distances, key=distances.get)
        if predicted == cls_label:
            correct += 1
        total += 1

option_c_accuracy = correct / total
print(f"Option C (Local Fine-Tuning Few-Shot FL) Accuracy: {option_c_accuracy:.4f}")


In [ ]:
print("=== DATASET VERIFICATION ===")

print("\nTRAIN DIR:", TRAIN_DIR)
print("Classes in TRAIN:", os.listdir(TRAIN_DIR))

print("\nTEST DIR:", TEST_DIR)
print("Classes in TEST:", os.listdir(TEST_DIR))

print("\nCLIENT DATASET SUMMARY:")
for client_id, (paths, labels) in CLIENT_DATASETS.items():
    unique_labels = set(labels)
    print(f"Client {client_id}:")
    print(f"  Number of samples: {len(paths)}")
    print(f"  Class labels: {unique_labels}")

print("\nFEW-SHOT DATA SAMPLE (Client 0):")
few_shot_sample = create_few_shot_client_data(CLIENT_DATASETS[0], k_shot=5)
print("  Total few-shot samples:", len(few_shot_sample))
print("  First 5 samples:", few_shot_sample[:5])
